In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import os

# 定义模型路径
base_model_path = "/kaggle/input/qwen-3/transformers/8b/1"
adapter_path = "/kaggle/input/map-hf-models12/exp166/"
save_path = "/kaggle/working/qwen-8b-merged-exp166"


tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.save_pretrained(save_path)
# 1. 在 CPU 上加载基础模型，使用 bfloat16 或 float16 以保持精度
#    注意：这里没有 device_map="auto"
print("Loading base model on CPU...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_path,
    num_labels=65, # 确保这里的 n_classes 是正确的
    torch_dtype=torch.bfloat16, # bfloat16 通常是 LLM 的首选
    trust_remote_code=True # Qwen 可能需要这个
)
base_model.config.pad_token_id = tokenizer.pad_token_id


# 2. 加载 LoRA 适配器
print("Loading PEFT adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. 在 CPU 上执行合并 (会比较慢，但不会OOM)
print("Merging model...")
model = model.merge_and_unload()
print("Merge complete.")

# 4. 保存合并后的模型和分词器
print(f"Saving merged model to {save_path}...")
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)



print("All done!")

2025-10-12 16:39:18.802206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760287159.032491      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760287159.095202      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading base model on CPU...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/qwen-3/transformers/8b/1 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading PEFT adapter...
Merging model...
Merge complete.
Saving merged model to /kaggle/working/qwen-8b-merged-exp166...
All done!
